# Train RDST on process A (train/test split) and save the model

* **Split:** the same as `train_test_split_process_A.ipynb`: batch-level, 30% test, stratified by label, `random_state=42`.
* **Model:** the `Model` class (RDST + spread + ridge, seed 0) from `03_rdst_spread_seed0.ipynb`. Its setup and model cells are
  run straight from that notebook, so the code is identical and stays in sync.
* **Output:** `models/rdst_process_A_seed0.pkl` (the fitted model, saved with cloudpickle so loading does not need the class
  definition) and `models/rdst_process_A_seed0.json` (split ids, chosen hyperparameters, test metrics).

In [1]:
import json
from pathlib import Path

import cloudpickle
import nbformat

# run the setup + model cells (1 and 2) of the RDST notebook in this namespace
rdst_nb = nbformat.read("03_rdst_spread_seed0.ipynb", as_version=4)
for i in (1, 2):
    exec(rdst_nb.cells[i].source)

/home/renku/work/input-data/Project 2 - BRPMM


## Split process A (same as `train_test_split_process_A.ipynb`)

In [2]:
folder = DATA_DIR / "process A" / "calibration data"
batches = load_batches(folder / "calibration_data.csv")
labels = pd.read_csv(folder / "calibration_labels.csv")
is_rejected = labels.set_index("batch_id").label.eq("rejected").astype(int)

train_ids, test_ids = train_test_split(labels.batch_id, test_size=0.30, random_state=42, stratify=labels.label)
train_ids, test_ids = train_ids.sort_index(), test_ids.sort_index()  # keep the file order, as in the RDST notebook
assert set(train_ids).isdisjoint(test_ids)
print(f"train: {len(train_ids)} batches ({is_rejected[train_ids].sum()} rejected) | "
      f"test: {len(test_ids)} batches ({is_rejected[test_ids].sum()} rejected)")

train: 112 batches (28 rejected) | test: 49 batches (12 rejected)


## Train on the training batches, check on the test batches

In [3]:
model = Model().fit(*inputs(batches, train_ids), is_rejected[train_ids].to_numpy())

pred = model.predict(*inputs(batches, test_ids))
true = is_rejected[test_ids].to_numpy()
metrics = {
    "f1_rejected": f1_score(true, pred), "precision": precision_score(true, pred),
    "recall": recall_score(true, pred), "accuracy": accuracy_score(true, pred),
    "rejected_caught": int(((pred == 1) & (true == 1)).sum()), "rejected_total": int(true.sum()),
    "false_alarms": int(((pred == 1) & (true == 0)).sum()), "accepted_total": int((true == 0).sum()),
}
print(" | ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}" for k, v in metrics.items()))

spread weight 0.5, alpha 0.001, CV F1 inside training 0.801


f1_rejected 0.833 | precision 0.833 | recall 0.833 | accuracy 0.918 | rejected_caught 10 | rejected_total 12 | false_alarms 2 | accepted_total 37


## Save the model

In [4]:
out = Path("models")
out.mkdir(exist_ok=True)
# Model is defined in __main__, so cloudpickle stores the class by value with the object
with open(out / "rdst_process_A_seed0.pkl", "wb") as f:
    cloudpickle.dump(model, f)

meta = {
    "process": "A", "seed": SEED, "length": LENGTH, "split": {"test_size": 0.30, "random_state": 42, "stratify": "label"},
    "spread_weight": model.weight, "alpha": float(model.alpha), "n_rdst_features": int(model.scalers[0].n_features_in_),
    "train_ids": list(train_ids), "test_ids": list(test_ids), "test_metrics": metrics,
}
(out / "rdst_process_A_seed0.json").write_text(json.dumps(meta, indent=2, default=float))
print("saved", sorted(p.name for p in out.iterdir()))

saved ['rdst_process_A_seed0.json', 'rdst_process_A_seed0.pkl']


## Check: load the saved model and predict again

In [5]:
with open(out / "rdst_process_A_seed0.pkl", "rb") as f:
    loaded = cloudpickle.load(f)
assert (loaded.predict(*inputs(batches, test_ids)) == pred).all()
print("reloaded model gives the same test predictions")

# to use it elsewhere: cloudpickle.load(open("models/rdst_process_A_seed0.pkl", "rb")).predict(series, spread_block),
# with series/spread_block built by inputs() from the RDST notebook (needs numpy, sklearn, aeon installed)

reloaded model gives the same test predictions
